In [1]:
#| include: false
# Disable tqdm progress bars; this is a hidden cell on the docs page
import os

os.environ["TQDM_DISABLE"] = "1"

# Measure the volume of a region

`compute_regions_volume` counts the voxels of every region in the label volume and stores the volumes, in mm³, on the region table as `ba.regions.volume`. With `cumsum=True` a region's volume includes the regions below it in the hierarchy.

In [2]:
import numpy as np
import pandas as pd

from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
br = ba.regions

ba.compute_regions_volume(cumsum=True)  # writes br.volume, in mm³
rows = np.isin(br.acronym, ['CA1', 'HPF', 'Isocortex'])
pd.DataFrame({'id': br.id[rows], 'acronym': br.acronym[rows],
              'volume_mm3': br.volume[rows].round(3)})

,id,acronym,volume_mm3
0,315,Isocortex,61.477
1,1089,HPF,21.318
2,382,CA1,5.105
3,-315,Isocortex,61.821
4,-1089,HPF,21.329
5,-382,CA1,5.104


Each region appears twice, once per hemisphere: the negative id is the left hemisphere; see [Tell left from right hemisphere](tell-left-from-right-hemisphere.ipynb). For the volume of both hemispheres together, add the two rows. Each call to `compute_regions_volume` replaces `br.volume`.

## Check by counting voxels

A region's volume is its number of voxels times the volume of one voxel, `(res_um / 1000) ** 3` mm³. Counting the voxels of HPF, selected as in [Get every voxel or coordinate inside a region](get-the-voxels-in-a-region.ipynb), gives the same volume as the two rows above added together:

In [3]:
ids = br.descendants(br.acronym2id('HPF'))['id']
indices = np.flatnonzero(np.isin(np.abs(br.id), ids))
n_voxels = np.isin(ba.label, indices).sum()

print('voxel count:          ', round(n_voxels * (ba.res_um / 1000) ** 3, 3))
print('compute_regions_volume:', round(br.volume[br.acronym == 'HPF'].sum(), 3))

voxel count:           42.647
compute_regions_volume: 42.647


## Without `cumsum`

By default, `cumsum=False`, each region counts only the voxels labelled with exactly that region. Most voxels are labelled with the finest regions, so a region with regions below it comes out small or zero. CA1 has its own voxels and keeps its volume:

In [4]:
ba.compute_regions_volume()
pd.DataFrame({'id': br.id[rows], 'acronym': br.acronym[rows],
              'volume_mm3': br.volume[rows].round(3)})

,id,acronym,volume_mm3
0,315,Isocortex,0.000
1,1089,HPF,0.211
2,382,CA1,5.105
3,-315,Isocortex,0.000
4,-1089,HPF,0.210
5,-382,CA1,5.104
